<a href="https://colab.research.google.com/github/tejasr9/QuotesApp/blob/main/final_backend_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import os
import re
import json
import requests
from datetime import datetime
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import shuffle
from sklearn.metrics.pairwise import cosine_similarity
from xgboost import XGBClassifier
import joblib
import requests
import time
import glob

In [8]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [9]:
API_KEY = "pub_35e774997e6c4a41b98cf031cd46dcaa"

def fetch_latest_news_multi(max_per_category=40):
    """
    Fetches news from multiple categories, handling pagination to get more results.
    """
    categories = [
        "politics", "world", "business", "sports",
        "technology", "entertainment", "science", "health", "general"
    ]
    all_articles = []

    for cat in categories:
        articles_for_cat = []
        next_page_token = None

        # Loop to handle multiple pages for the current category
        while len(articles_for_cat) < max_per_category:
            # Construct the URL. Add the 'page' parameter if we have a token.
            base_url = f"https://newsdata.io/api/1/news?apikey={API_KEY}&category={cat}&language=en&country=us"
            url = f"{base_url}&page={next_page_token}" if next_page_token else base_url

            try:
                r = requests.get(url)
                r.raise_for_status() # Raise an error for bad responses
                data = r.json()

                if "results" in data and data["results"]:
                    titles = [a["title"] for a in data["results"] if a.get("title")]
                    articles_for_cat.extend(titles)

                    # Get the token for the next page
                    next_page_token = data.get("nextPage")
                    if not next_page_token:
                        # No more pages for this category, so break the inner loop
                        break
                else:
                    # No results found for this page, stop for this category
                    break

            except requests.exceptions.RequestException as e:
                print(f"Error fetching {cat}: {e}")
                break # Stop trying this category if an error occurs

            time.sleep(0.1) # Be polite to the API

        # Add the collected articles for this category to the main list
        all_articles.extend(articles_for_cat)
        print(f"Collected {len(articles_for_cat)} headlines for '{cat}'. Total: {len(all_articles)}")

    print(f"✅ Collected {len(all_articles)} total API headlines.")
    return all_articles


def cache_api_news(articles):
    date = datetime.now().strftime("%Y-%m-%d")
    filename = f"api_news_{date}.json"

    # If file exists, load existing articles
    if os.path.exists(filename):
        with open(filename, "r") as f:
            existing = json.load(f)
    else:
        existing = []

    # Merge and remove duplicates
    all_articles = list(set(existing + articles))

    with open(filename, "w") as f:
        json.dump(all_articles, f)

    print(f"💾 Cached {len(all_articles)} unique articles for {date}")


def load_cached_news():
    date = datetime.now().strftime("%Y-%m-%d")
    try:
        with open(f"api_news_{date}.json") as f:
            print(f"📂 Loaded cached API news for {date}")
            return json.load(f)
    except FileNotFoundError:
        print("⚠️ No cache found. Fetching new data...")
        return []



def load_multiple_caches(days=3):
    """
    Load cached API news from the last 'n' days.
    Example: loads api_news_2025-10-09.json, api_news_2025-10-10.json, api_news_2025-10-11.json
    """
    files = sorted(glob.glob("api_news_*.json"))[-days:]
    articles = []
    for f in files:
        with open(f, "r") as file:
            data = json.load(file)
            articles.extend(data)
    print(f"📅 Loaded {len(articles)} articles from last {len(files)} days of cache.")
    return articles


def load_dataset(path):
    df = pd.read_parquet(path)
    df["content"] = (df["title"].fillna("") + " " + df["text"].fillna("")).apply(clean_text)
    df = df[["content", "label"]]
    df["label"] = df["label"].astype(int)
    return df



def train_model(dataset_path):
    df = load_dataset(dataset_path)

    # Load last 3 days of cached API news
    api_articles = load_multiple_caches(days=3)

    # If no cache found yet, fetch fresh
    if not api_articles:
        api_articles = fetch_latest_news_multi()
        cache_api_news(api_articles)
    df_api = pd.DataFrame({"content": [clean_text(a) for a in api_articles], "label": 1})

    # --- Merge and shuffle
    df_all = shuffle(pd.concat([df, df_api], ignore_index=True), random_state=42)
    print("🧩 Dataset size after merging:", df_all.shape)

    # --- TF-IDF Feature Extraction
    tfidf = TfidfVectorizer(
        max_features=30000,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.9,
        stop_words="english"
    )
    X = tfidf.fit_transform(df_all["content"])
    y = df_all["label"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    # --- XGBoost Model + Calibration
    base_model = XGBClassifier(
        n_estimators=250,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        n_jobs=-1
    )
    calibrated_model = CalibratedClassifierCV(base_model, method="isotonic", cv=3)
    calibrated_model.fit(X_train, y_train)

    # --- Evaluation
    preds = calibrated_model.predict(X_test)
    print("\n📊 Classification Report:\n", classification_report(y_test, preds))

    # --- Save model + vectorizer
    joblib.dump(calibrated_model, "xgb_model.pkl")
    joblib.dump(tfidf, "tfidf_vectorizer.pkl")
    print("💾 Model & vectorizer saved successfully!")



In [10]:
#FINAL HYBRID PREDICTION FUNCTION
def predict_news_with_api(user_input, model, tfidf, clean_text, threshold=0.6):
    """
    Predict whether given news is REAL or FAKE:
      1️⃣ If semantically similar to latest API news -> Real
      2️⃣ Else use model prediction
    """
    # Step 1: Load cached API data
    api_articles = load_cached_news()
    if not api_articles:
        api_articles = fetch_latest_news_multi()
        cache_api_news(api_articles)

    api_clean = [clean_text(t) for t in api_articles]
    X_api = tfidf.transform(api_clean)

    # Step 2: Clean & vectorize user input
    user_clean = clean_text(user_input)
    X_user = tfidf.transform([user_clean])

    # Step 3: Check similarity
    sim_scores = cosine_similarity(X_user, X_api)
    max_sim = sim_scores.max()

    if max_sim > threshold:
        return {
            "input": user_input,
            "label": "REAL (Matched with latest verified news)",
            "confidence": round(max_sim * 100, 2),
            "source": "API Similarity"
        }

    # Step 4: Model prediction
    prob = model.predict_proba(X_user)[0][1]
    label = "REAL" if prob >= 0.45 else "FAKE"
    return {
        "input": user_input,
        "label": label,
        "confidence": round(prob * 100, 2),
        "source": "XGBoost Model"
    }

In [ ]:
if __name__ == "__main__":

    latest_articles = fetch_latest_news_multi()
    cache_api_news(latest_articles)

    # Train (run once a week or when dataset updates)
    train_model("news_dataset.parquet")

    # Load model & vectorizer
    model = joblib.load("xgb_model.pkl")
    tfidf = joblib.load("tfidf_vectorizer.pkl")

    # Test Predictions
    sample_news = [
        " In a shocking revelation that has rocked the scientific community and sent confectioners into a frenzy, a team of international geophysicists at CERN announced today that new seismic data unequivocally proves the Earth's inner core is not, as previously believed, a superheated ball of solid iron, but rather a colossal, molten mass of cosmic-grade marshmallows.",
        "The Defense Cyber Security market is estimated to grow at a CAGR of 7.7% from 2022 to 2031, owing to the growing demand for defense-system to be present against cyber-attacks across the globe. In accordance with segmentation, “by type, the Endpoint security solutions segment dominated the global Defense Cyber Security market in 2021, in terms of revenue. By deployment, the on-premises segment dominated the global defense cyber security market in 2021, in terms of revenue. By solution, the identity and access management segment application, the military segment dominated the global Defense Cyber Security market in 2021, in terms of revenue. Presently, North America is the highest revenue contributor and expected to lead the market during the forecast period, followed by Europe."
    ]

    for news in sample_news:
        result = predict_news_with_api(news, model, tfidf, clean_text)
        print("\n📰", result["input"])
        print("→ Prediction:", result["label"])
        print("→ Confidence:", result["confidence"], "%")
        print("→ Source:", result["source"])


Error fetching politics: 429 Client Error: TOO MANY REQUESTS for url: https://newsdata.io/api/1/news?apikey=pub_35e774997e6c4a41b98cf031cd46dcaa&category=politics&language=en&country=us
Collected 0 headlines for 'politics'. Total: 0
Error fetching world: 429 Client Error: TOO MANY REQUESTS for url: https://newsdata.io/api/1/news?apikey=pub_35e774997e6c4a41b98cf031cd46dcaa&category=world&language=en&country=us
Collected 0 headlines for 'world'. Total: 0
Error fetching business: 429 Client Error: TOO MANY REQUESTS for url: https://newsdata.io/api/1/news?apikey=pub_35e774997e6c4a41b98cf031cd46dcaa&category=business&language=en&country=us
Collected 0 headlines for 'business'. Total: 0
Error fetching sports: 429 Client Error: TOO MANY REQUESTS for url: https://newsdata.io/api/1/news?apikey=pub_35e774997e6c4a41b98cf031cd46dcaa&category=sports&language=en&country=us
Collected 0 headlines for 'sports'. Total: 0
Error fetching technology: 429 Client Error: TOO MANY REQUESTS for url: https://new